# FedFlower Phase 1 — Centralized CNN Baseline

> **Before running:** Go to `Runtime → Change Runtime Type → T4 GPU → Save`

## Cell 1 — Install Libraries
Run this every time you open a new Colab session.

In [ ]:
!pip install torch torchvision torchaudio --index-url https://download.pytorch.org/whl/cu118 -q
!pip install matplotlib numpy pillow scikit-learn grad-cam -q
!pip install tensorflow onnx onnxruntime -q
print("✅ All libraries installed")

## Cell 2 — Verify GPU

In [ ]:
import torch
print("CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
else:
    print("⚠️  No GPU! Go to Runtime → Change Runtime Type → T4 GPU")

## Cell 3 — Download Oxford 102 Flowers Dataset (~330 MB, takes 2-3 min first time)

In [ ]:
import torchvision.datasets as datasets
import torchvision.transforms as transforms
from torch.utils.data import DataLoader

# Training transform — RandomResizedCrop preserves aspect ratio,
# matching the Resize(256)+CenterCrop(224) used at inference time
train_transform = transforms.Compose([
    transforms.RandomResizedCrop(224, scale=(0.7, 1.0)),
    transforms.RandomHorizontalFlip(),
    transforms.RandomRotation(15),
    transforms.ColorJitter(brightness=0.3, contrast=0.3, saturation=0.3, hue=0.1),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
])

# Test transform — Resize(256)+CenterCrop(224), preserves aspect ratio,
# identical to Phase 3 evaluation and the Android app
test_transform = transforms.Compose([
    transforms.Resize(256),
    transforms.CenterCrop(224),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
])

print("Downloading Oxford 102 Flowers...")
train_data = datasets.Flowers102(root='./data', split='train', download=True, transform=train_transform)
val_data   = datasets.Flowers102(root='./data', split='val',   download=True, transform=test_transform)
test_data  = datasets.Flowers102(root='./data', split='test',  download=True, transform=test_transform)

train_loader = DataLoader(train_data, batch_size=32, shuffle=True,  num_workers=2, pin_memory=True)
val_loader   = DataLoader(val_data,   batch_size=32, shuffle=False, num_workers=2, pin_memory=True)
test_loader  = DataLoader(test_data,  batch_size=32, shuffle=False, num_workers=2, pin_memory=True)

print(f"✅ Train: {len(train_data)} | Val: {len(val_data)} | Test: {len(test_data)}")

## Cell 4 — Visualize Sample Images

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

fig, axes = plt.subplots(2, 5, figsize=(15, 6))
fig.suptitle("Oxford 102 Flowers — Sample Images", fontsize=14, fontweight='bold')
for i, ax in enumerate(axes.flat):
    img, label = train_data[i * 80]
    img = img.numpy().transpose(1, 2, 0)
    img = img * [0.229, 0.224, 0.225] + [0.485, 0.456, 0.406]
    img = np.clip(img, 0, 1)
    ax.imshow(img)
    ax.set_title(f'Class {label + 1}', fontsize=9)
    ax.axis('off')
plt.tight_layout()
plt.savefig('sample_flowers.png', dpi=150)
plt.show()
print("✅ Saved sample_flowers.png")

## Cell 5 — Build the CNN Model (ResNet50 + Transfer Learning)

**Why ResNet50?** Pre-trained on 1.2M ImageNet images — already knows edges, textures, shapes. We freeze early layers and only train the final flower-specific head. This is *transfer learning*.

In [ ]:
import torch
import torch.nn as nn
import torchvision.models as models

class FlowerCNN(nn.Module):
    def __init__(self, num_classes=102):
        super(FlowerCNN, self).__init__()
        self.backbone = models.resnet50(weights='IMAGENET1K_V2')
        # Freeze early layers — they already detect edges/textures
        for name, param in self.backbone.named_parameters():
            if 'layer4' not in name and 'fc' not in name:
                param.requires_grad = False
        # Replace final classifier for 102 flower classes
        in_features = self.backbone.fc.in_features
        self.backbone.fc = nn.Sequential(
            nn.Linear(in_features, 512),
            nn.BatchNorm1d(512),
            nn.ReLU(inplace=True),
            nn.Dropout(0.4),
            nn.Linear(512, num_classes)
        )

    def forward(self, x):
        return self.backbone(x)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model  = FlowerCNN(num_classes=102).to(device)
trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
total     = sum(p.numel() for p in model.parameters())
print(f"✅ Model on: {device}")
print(f"   Trainable: {trainable:,} / {total:,} params ({100*trainable/total:.1f}%)")

## Cell 6 — Training Setup

In [ ]:
import torch.optim as optim
from torch.optim.lr_scheduler import CosineAnnealingLR

criterion = nn.CrossEntropyLoss(label_smoothing=0.1)
optimizer = optim.Adam(filter(lambda p: p.requires_grad, model.parameters()), lr=1e-4, weight_decay=1e-4)
scheduler = CosineAnnealingLR(optimizer, T_max=30)

def train_epoch(model, loader, optimizer, criterion, device):
    model.train()
    total_loss, correct, total = 0, 0, 0
    for imgs, labels in loader:
        imgs, labels = imgs.to(device), labels.to(device)
        optimizer.zero_grad()
        outputs = model(imgs)
        loss = criterion(outputs, labels)
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
        optimizer.step()
        total_loss += loss.item()
        _, preds = outputs.max(1)
        correct += preds.eq(labels).sum().item()
        total   += labels.size(0)
    return total_loss / len(loader), 100.0 * correct / total

def evaluate(model, loader, criterion, device):
    model.eval()
    total_loss, correct, total = 0, 0, 0
    with torch.no_grad():
        for imgs, labels in loader:
            imgs, labels = imgs.to(device), labels.to(device)
            outputs = model(imgs)
            loss = criterion(outputs, labels)
            total_loss += loss.item()
            _, preds = outputs.max(1)
            correct += preds.eq(labels).sum().item()
            total   += labels.size(0)
    return total_loss / len(loader), 100.0 * correct / total

print("✅ Training functions ready")

## Cell 7 — Train for 30 Epochs (~25 minutes on T4)

Expected: val accuracy reaches **85–92%** by epoch 30. Best model auto-saved as `best_model.pth`.

In [ ]:
EPOCHS = 30
best_val_acc = 0
history = {'train_loss': [], 'train_acc': [], 'val_acc': []}

print(f"{'Epoch':>6} {'Loss':>10} {'Train':>9} {'Val':>9} {'Best':>6}")
print("─" * 48)

for epoch in range(EPOCHS):
    train_loss, train_acc = train_epoch(model, train_loader, optimizer, criterion, device)
    val_loss, val_acc     = evaluate(model, val_loader, criterion, device)
    scheduler.step()

    history['train_loss'].append(train_loss)
    history['train_acc'].append(train_acc)
    history['val_acc'].append(val_acc)

    is_best = val_acc > best_val_acc
    if is_best:
        best_val_acc = val_acc
        torch.save(model.state_dict(), 'best_model.pth')

    mark = " ★" if is_best else ""
    print(f"{epoch+1:>6} {train_loss:>10.4f} {train_acc:>8.2f}% {val_acc:>8.2f}%{mark}")

print(f"\n✅ Done! Best val accuracy: {best_val_acc:.2f}%  →  saved as best_model.pth")

## Cell 8 — Plot Training History

In [ ]:
import matplotlib.pyplot as plt

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))
ax1.plot(history['train_loss'], color='#E63946')
ax1.set_title('Training Loss'); ax1.set_xlabel('Epoch'); ax1.grid(True, alpha=0.3)
ax2.plot(history['train_acc'], label='Train', color='#1F4E79')
ax2.plot(history['val_acc'],   label='Val',   color='#2D9CDB')
ax2.set_title('Accuracy'); ax2.set_xlabel('Epoch'); ax2.legend(); ax2.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig('training_history.png', dpi=150)
plt.show()

## Cell 9 — Final Test Set Evaluation

**Write down the test accuracy** — you will compare it against federated learning in Notebook 2.

In [ ]:
from sklearn.metrics import f1_score
import numpy as np, json

model.load_state_dict(torch.load('best_model.pth'))
model.eval()
all_preds, all_labels = [], []

with torch.no_grad():
    for imgs, labels in test_loader:
        imgs = imgs.to(device)
        _, preds = model(imgs).max(1)
        all_preds.extend(preds.cpu().numpy())
        all_labels.extend(labels.numpy())

test_acc = 100.0 * np.sum(np.array(all_preds) == np.array(all_labels)) / len(all_labels)
f1       = f1_score(all_labels, all_preds, average='macro')

print("=" * 45)
print(f"CENTRALIZED CNN — FINAL TEST RESULTS")
print("=" * 45)
print(f"Top-1 Accuracy : {test_acc:.2f}%")
print(f"Macro F1-Score : {f1:.4f}")
print("=" * 45)
print("⬇️  Download best_model.pth from the Files panel (left sidebar) before closing Colab!")

with open('centralized_results.json', 'w') as f:
    json.dump({"centralized_test_acc": test_acc, "centralized_f1": f1}, f, indent=2)